# R7.0-PY｜ADT V6 卫星高度计独立流轴验证 — 单 Notebook 全流程

**运行条件：WSL Python 3.10+ / VS Code Notebook；无需 MATLAB、无 `.bat` 文件、无需重训。**

本 Notebook 从 1993–2023 年 F 盘每日 ADT NetCDF 直接计算卫星参考流轴，并与已冻结的 GLORYS12V1 50 m 监测流轴和**实际存在的**模型预测路径比较；自动进行 **A、B、PN、C、D、E 六断面**误差、月尺度距平和长期趋势、Bootstrap 置信区间、中英文图表以及 ZIP+SHA256 归档。

**注意：这是用户 MATLAB V6 方法的 Python 数值移植，不是经 MATLAB 逐日严格数值等价性认证的二进制复刻。** 原 V6 的台湾连续入口种子（DP）、双尺度梯度、闭合等值线惩罚、ridge-band 候选、Tokara/Kyushu 及台湾东北角门限、失败时局部回捕和 3 步 mini-track、两轮受脊线约束拐角修正已实现。SciPy/contourpy 与 MATLAB 的等值线/滤波边缘可能出现数值差别。**正式投稿前，建议用原 MATLAB 参考输出抽样核对，并人工检查异常日期。**

- 默认 `RUN_FULL=True`：自动跑 6 天冒烟试验，随后全量扫描并断点继续执行 1993–2023 年 ADT 数据，不用手动切换 Notebook。
- 逐日原始 NetCDF 不复制到输出目录；只读 120–150°E、20–40°N 的子区域，不存每日大图；只保存流轴和质量日志。
- **监测验证与预测验证分开**。如本地尚无带起报日/目标日的真实预测路径，预测部分明确记为“待提供预测路径”，不会伪造模型预测。
- 1993–2023 是 ADT 当前提供的时间范围；2024 年数据不纳入本轮 ADT 独立验证。


In [ ]:
# ===== Cell 0：配置、环境依赖、安装缺失依赖（仅第一次需要） =====
from pathlib import Path
from dataclasses import dataclass,asdict
import os, sys, json, re, time, gc, traceback, hashlib, subprocess, importlib.util, warnings
import numpy as np
import pandas as pd
# 保持单文件可运行，依赖仅安装在当前 Notebook 内核
required={'xarray':'xarray','scipy':'scipy','pandas':'pandas','contourpy':'contourpy','matplotlib':'matplotlib','netCDF4':'netCDF4','tqdm':'tqdm'}
missing=[pip for mod,pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing Python dependencies:',missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from tqdm.auto import tqdm
from IPython.display import display,FileLink

@dataclass
class CFG:
    ROOT:str='/mnt/d/KuroshioDaily1993_2025'
    ADT_ROOT:str='/mnt/f/01-CMEMS/SEALEVEL_GLO_PHY_L4_MY_008_047/cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D'
    GLORYS_AXIS_SUBDIR:str='final/tracks_final'
    START_DATE:str='1993-01-01'
    END_DATE:str='2023-12-31'
    RUN_FULL:bool=True            # True = 冒烟通过自动处理整个1993—2023，不需再切换
    SMOKE_DATES:tuple=('1993-01-15','2000-07-15','2010-01-15','2017-09-15','2020-06-15','2023-06-15')
    FORCE_RETRACK:bool=False      # False = 断点续跑，默认不覆盖已成功的ADT流轴
    RETRY_FAILED:bool=False       # True = 重试以往失败日，否则留下原错误日志
    USE_TOPOLOGY:bool=True        # 保留原V6闭合等值线惩罚；勿在正式比较中随意关闭
    RIDGE_REGULARIZATION:bool=True
    ZIP_EVERY_N_DAYS:int=500     # 仅归档轻量CSV日志，非每日.nc/npz
    PRINT_EVERY:int=50
    MIN_MONTH_DAYS:int=15
    BOOTSTRAP_N:int=400
    MIN_TERMINAL_LON:float=143.
    COMPARE_MAT_REFERENCE:bool=False
    MATLAB_REFERENCE_DIR:str=''  # 如可得到MATLAB原版某些日的.mat，启用独立对照
    FORECAST_MANIFEST:str=''     # 留空=自动查找以前R70输出中的predictions_manifest.csv
    RANDOM_SEED:int=20261015
cfg=CFG()
ROOT=Path(cfg.ROOT)
ADT_ROOT=Path(cfg.ADT_ROOT)
GLORYS=ROOT/cfg.GLORYS_AXIS_SUBDIR
RUN=ROOT/'R70_ADT_Independent_Python'
ADT_AX=RUN/'adt_axes_npz';META=RUN/'adt_track_meta';REP=RUN/'reports';OUT=RUN/'results'
for x in [RUN,ADT_AX,META,REP,OUT]:x.mkdir(parents=True,exist_ok=True)
if not ADT_ROOT.is_dir():
    raise FileNotFoundError(f'F盘ADT目录未找到: {ADT_ROOT}\nWSL可使用: sudo mkdir -p /mnt/f && sudo mount -t drvfs F: /mnt/f')
if not GLORYS.is_dir():
    raise FileNotFoundError('原GLORYS冻结流轴目录不存在: '+str(GLORYS))
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False));print('Python:',sys.version)


## 第一阶段：嵌入式 Python ADT V6 流轴追踪器
以下两个单元包含**完整可运行的 Python 算法代码**，并没有引用目录外的 `.py`。已采用与 MATLAB V6 对应的数值与物理规则，保留原轨迹和修正后的轨迹。

In [ ]:
"""Independent satellite ADT Kuroshio axis V6 Python port.
Port of user-supplied kuroshio_axis_ADT_final_v4_seedfix_v6.m, 2026-10.
Topological penalty, seed DP, ridge-band, local multistep recovery, and ridge-constrained corners retained.
Numerical equivalence is NOT assumed; validate with MATLAB on same snapshots before quantitative publication.
"""
from __future__ import annotations
import math, os, re, warnings, traceback
from dataclasses import dataclass
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd
import xarray as xr
from scipy.ndimage import gaussian_filter, convolve
from scipy.interpolate import RegularGridInterpolator
import contourpy

KMLAT=111.32

def geo(x,y,x2,y2):
    return np.hypot((np.asarray(x2)-x)*KMLAT*np.cos(np.deg2rad((np.asarray(y2)+y)/2)),(np.asarray(y2)-y)*KMLAT)
def dist_points(x,y,xp,yp): return geo(x,y,xp,yp)
def km2deg(dx,dy,lat):
    return np.asarray(dx)/max(1e-9,KMLAT*math.cos(math.radians(float(lat)))),np.asarray(dy)/KMLAT
def tangent(x1,y1,x2,y2,lat): return np.array([(x2-x1)*KMLAT*np.cos(np.deg2rad(lat)),(y2-y1)*KMLAT],float)
def unit(a):
    a=np.asarray(a,float);n=np.linalg.norm(a)
    return a/n if n>1e-10 and np.isfinite(n) else np.array([np.nan,np.nan])
def cosangle(a,b): return float(np.clip(np.dot(a,b),-1,1))
def ang(a,b):return math.degrees(math.acos(cosangle(a,b)))
def turn3(x1,y1,x2,y2,x3,y3):
    v1=unit(tangent(x1,y1,x2,y2,(y1+y2)/2));v2=unit(tangent(x2,y2,x3,y3,(y2+y3)/2))
    return ang(v1,v2) if np.isfinite(v1).all() and np.isfinite(v2).all() else np.nan
def turning(x,y):
    z=np.full(len(x),np.nan)
    for i in range(1,len(x)-1):z[i]=turn3(x[i-1],y[i-1],x[i],y[i],x[i+1],y[i+1])
    return z

def ridge_band(g,valid,sampling=3.,band=.8,global_ratio=.55,neighbor=15.):
    g=np.asarray(g,float);valid=np.asarray(valid,bool);n=len(g)
    if not valid.any():return np.array([],int)
    peaks=[]
    if n>=2:
        if valid[0] and (not valid[1] or g[0]>=g[1]):peaks.append(0)
        if valid[-1] and (not valid[-2] or g[-1]>=g[-2]):peaks.append(n-1)
    for j in range(1,n-1):
        if valid[j-1] and valid[j] and valid[j+1] and g[j]>=g[j-1] and g[j]>=g[j+1]:peaks.append(j)
    gmax=np.max(g[valid]);
    if not np.isfinite(gmax) or gmax<=0:return np.array([],int)
    mask=valid&(g>=global_ratio*gmax);rad=max(1,int(round(neighbor/max(sampling,1e-8))))
    for j in peaks:
        idx=np.arange(max(0,j-rad),min(n,j+rad+1));mask[idx]|=valid[idx]&(g[idx]>=band*g[j])
    cand=np.flatnonzero(mask)
    return cand if len(cand) else np.array([np.flatnonzero(valid)[np.argmax(g[valid])]])

class Fields:
    def __init__(self,lon,lat,a,sigma_fine=1.,sigma_coarse=3.,build_topo=True):
        self.lon=np.asarray(lon,float);self.lat=np.asarray(lat,float)
        self.A=np.asarray(a,float)
        assert self.A.shape==(len(lat),len(lon)),(self.A.shape,len(lat),len(lon))
        assert np.all(np.diff(lon)>0) and np.all(np.diff(lat)>0)
        valid=np.isfinite(self.A)
        def filt(sig):
            num=gaussian_filter(np.where(valid,self.A,0),sig,mode='nearest')
            den=gaussian_filter(valid.astype(float),sig,mode='nearest')
            return np.where(den>=.6,num/np.maximum(den,1e-12),np.nan)
        self.fine=filt(sigma_fine);self.coarse=filt(sigma_coarse)
        self.gx=self.gy=self.g=self.gc=None
        # MATLAB gradient dA/dlon, dA/dlat, converted m/km.
        dy,dx=np.gradient(self.fine,self.lat,self.lon)
        dyc,dxc=np.gradient(self.coarse,self.lat,self.lon)
        gridcos=np.maximum(.1,np.cos(np.deg2rad(self.lat[:,None])))
        self.gx=dx/(KMLAT*gridcos);self.gy=dy/KMLAT
        self.g=np.hypot(self.gx,self.gy)
        self.gc=np.hypot(dxc/(KMLAT*gridcos),dyc/KMLAT)
        self.g95=max(float(np.nanpercentile(self.g,95)),1e-10)
        self.gc95=max(float(np.nanpercentile(self.gc,95)),1e-10)
        self.i={}
        for key,v in [('a',self.fine),('g',self.g),('gc',self.gc),('gx',self.gx),('gy',self.gy)]:
            self.i[key]=RegularGridInterpolator((self.lat,self.lon),v,bounds_error=False,fill_value=np.nan)
        self.topo=Topology(self.lon,self.lat,self.fine) if build_topo else None
    def get(self,name,ys,xs):
        yy,xx=np.broadcast_arrays(np.asarray(ys,float),np.asarray(xs,float))
        z=self.i[name](np.column_stack([yy.ravel(),xx.ravel()])).reshape(yy.shape)
        return float(z) if z.ndim==0 else z
    def inside(self,x,y):return self.lon[0]<=x<=self.lon[-1] and self.lat[0]<=y<=self.lat[-1]
    def profile(self,x0,y0,normal,width,sampling,attrs=('g','gc','a','gx','gy')):
        off=np.arange(-width,width+1e-7,sampling)
        dx,dy=km2deg(off*normal[0],off*normal[1],y0)
        xx=x0+dx;yy=y0+dy
        results={k:self.get(k,yy,xx) for k in attrs}
        return off,xx,yy,results

class Topology:
    """MATLAB contourc-like level library at 2-cm increments, small shapes only."""
    def __init__(self,lon,lat,z,step=.02,minlen=80.):
        finite=z[np.isfinite(z)]
        self.step=step;self.segments={}
        if len(finite)<10:return
        lev1=np.ceil(finite.min()/step)*step;lev2=np.floor(finite.max()/step)*step
        if lev2-lev1<step:return
        cg=contourpy.contour_generator(x=lon,y=lat,z=np.ma.masked_invalid(z),name='serial',corner_mask=False)
        levels=np.arange(lev1,lev2+step*.2,step)
        for li,v in enumerate(levels):
            seg=[]
            for xy in cg.lines(float(v)):
                if len(xy)<3:continue
                x,y=xy[:,0],xy[:,1]
                length=np.sum(geo(x[:-1],y[:-1],x[1:],y[1:]))
                if length<minlen:continue
                seg.append((xy[::3].copy(),bool(geo(x[0],y[0],x[-1],y[-1])<25)))
            if seg:self.segments[li]=seg
        self.start=float(lev1);self.n=len(levels)
    def query(self,x,y,a,match=18.):
        if not np.isfinite(a) or not hasattr(self,'n'):return False,False
        ind=int(np.clip(np.round((a-self.start)/self.step),0,self.n-1))
        if ind not in self.segments:return False,False
        best=float('inf');closed=False
        for pts,isclosed in self.segments[ind]:
            d=float(np.min(geo(x,y,pts[:,0],pts[:,1])))
            if d<best:best=d;closed=isclosed
        return bool(closed), bool(best<=match)

@dataclass
class V6:
    max_points:int=650
    stop_lon:float=144.3
    stop_lat:float=29.0
    use_topology:bool=True
    regularize:bool=True
    enable_recovery:bool=True
    # Original MATLAB V6 parameters retained in port; do not tune on ADT–GLORYS agreement.

GATES=np.array([[128,131.5,27,31.8],[133,138.5,27,34.8],[140,144.8,29,37.5]])
def gate_pass(x,y,g,margin):return x>=g[1]-margin and g[2]-.8<=y<=g[3]+.8
def guide(x,y,stage):
    if stage>=len(GATES):return np.array([1.,0.])
    a,b,c,d=GATES[stage];xt=np.clip(x,a,b);yt=np.clip(y,c,d)
    v=np.array([(xt-x)*KMLAT*np.cos(np.deg2rad(y)),(yt-y)*KMLAT])
    return unit(v) if np.linalg.norm(v)>1e-6 else np.array([1.,0.])

def taiwan_seed(F):
    latidx=np.where((F.lat>=22)&(F.lat<=24.4))[0]
    lonidx=np.where((F.lon>=121)&(F.lon<=124))[0]
    if len(latidx)<6 or len(lonidx)<3:raise RuntimeError('insufficient Taiwan seed grid')
    C=[];GG=F.g[np.ix_(latidx,lonidx)];GC=F.gc[np.ix_(latidx,lonidx)]
    gs=max(np.nanpercentile(GG,90),1e-10);gcs=max(np.nanpercentile(GC,90),1e-10)
    for iy in latidx:
        g=F.g[iy,lonidx]
        valid=np.isfinite(g)&np.isfinite(F.gc[iy,lonidx])&np.isfinite(F.fine[iy,lonidx])&np.isfinite(F.gx[iy,lonidx])&np.isfinite(F.gy[iy,lonidx])
        peaks=np.zeros(len(lonidx),bool)
        peaks[1:-1]=(g[1:-1]>=g[:-2])&(g[1:-1]>=g[2:])
        jj=np.flatnonzero(valid&peaks)
        if not len(jj):jj=np.flatnonzero(valid)
        rows=[]
        for j in jj:
            ix=lonidx[j];gxx=F.gx[iy,ix];gyy=F.gy[iy,ix];ng=np.hypot(gxx,gyy)
            if ng<=1e-10 or gxx/ng<.2:continue
            closed,known=F.topo.query(F.lon[ix],F.lat[iy],F.fine[iy,ix]) if F.topo else (False,False)
            penalty=2.5 if closed and known else (.15 if not known else 0)
            score=min(F.g[iy,ix]/gs,1.8)+.75*min(F.gc[iy,ix]/gcs,1.8)+.35*gxx/ng-penalty
            rows.append([F.lon[ix],F.lat[iy],F.g[iy,ix],F.gc[iy,ix],F.fine[iy,ix],gxx/ng,score])
        C.append(sorted(rows,key=lambda a:-a[6])[:6])
    S=[];P=[];L=[]
    for ri,cur in enumerate(C):
        sco=np.array([z[6] for z in cur],float);back=np.full(len(cur),-1,int);plen=np.ones(len(cur),int)
        if ri and len(C[ri-1]):
            for j,c in enumerate(cur):
                for k,p in enumerate(C[ri-1]):
                    dx=abs(c[0]-p[0])*KMLAT*np.cos(np.deg2rad((c[1]+p[1])/2))
                    if dx>42:continue
                    d=np.hypot(dx,KMLAT*abs(c[1]-p[1]))
                    trans=.85*(dx/42)**2+.30*(abs(c[4]-p[4])/.08)**2+.05*(d/20)**2
                    v=S[-1][k]+c[6]-trans
                    if v>sco[j]:sco[j]=v;back[j]=k;plen[j]=L[-1][k]+1
        S.append(sco);P.append(back);L.append(plen)
    best=(-np.inf,-1,-1)
    for i,Ci in enumerate(C):
        for j in range(len(Ci)):
            if L[i][j]>=6 and S[i][j]+.35*L[i][j]>best[0]:best=(S[i][j]+.35*L[i][j],i,j)
    if best[1]<0:raise RuntimeError('V6 Taiwan entrance DP no 6-point continuous northward ridge')
    i,j=best[1:];seg=[]
    for _ in range(int(L[i][j])):
        seg.append(C[i][j][:2]);j0=P[i][j];i-=1;j=j0
        if j<0:break
    seg=np.asarray(seg[::-1],float)
    if len(seg)<6:raise RuntimeError('short Taiwan entrance seed')
    return seg

def support_ahead(F,x,y,t,step=12.,width=28.,spacing=4.):
    dx,dy=km2deg(step*t[0],step*t[1],y);xp=x+dx;yp=y+dy
    if not F.inside(xp,yp):return 0.
    _,xx,yy,z=F.profile(xp,yp,[-t[1],t[0]],width,spacing,('g','gc'))
    valid=np.isfinite(z['g'])&np.isfinite(z['gc'])&(z['g']>0)
    if not valid.any():return 0.
    scores=.55*np.minimum(z['g'][valid]/F.g95,1.5)+.45*np.minimum(z['gc'][valid]/F.gc95,1.5)
    return float(np.max(scores))
def support_multi(F,x,y,t,nsteps=3,step=12.,width=30.,spacing=4.):
    vals=[]
    for _ in range(nsteps):
        dx,dy=km2deg(step*t[0],step*t[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):return 0.
        _,xx,yy,z=F.profile(xp,yp,[-t[1],t[0]],width,spacing,('g','gc'))
        good=np.isfinite(z['g'])&np.isfinite(z['gc'])&(z['g']>0)
        if not good.any():return 0.
        cand=np.flatnonzero(good)
        support=.55*np.minimum(z['g'][cand]/F.g95,1.5)+.45*np.minimum(z['gc'][cand]/F.gc95,1.5)
        j=cand[np.argmax(support)];vals.append(np.max(support));x,y=xx[j],yy[j]
    return float(min(vals))
def mini_track(F,x,y,t):
    xstart,ystart=x,y;vals=[]
    for _ in range(3):
        dx,dy=km2deg(14*t[0],14*t[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):return 0.,-np.inf
        off,xx,yy,z=F.profile(xp,yp,[-t[1],t[0]],34,4)
        ok=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['gx'])&np.isfinite(z['gy'])&(z['g']>0)
        cand=ridge_band(z['g'],ok,4,.76,.45,20)
        best=(-np.inf,None,None,None)
        for j in cand:
            tc=unit([-z['gy'][j],z['gx'][j]])
            if not np.isfinite(tc).all() or cosangle(t,tc)<np.cos(np.deg2rad(100)):continue
            sr=.55*min(z['g'][j]/F.g95,1.5)+.45*min(z['gc'][j]/F.gc95,1.5)
            score=sr+.2*cosangle(t,tc)-.12*(off[j]/34)**2
            if score>best[0]:best=(score,j,tc,sr)
        if best[1] is None:return 0.,-np.inf
        j=best[1];x,y=xx[j],yy[j];vals.append(best[3]);t=unit(.45*t+.55*best[2])
    return float(min(vals)),float((x-xstart)*KMLAT*np.cos(np.deg2rad((y+ystart)/2)))

def recover(F,x,y,a0,t,g,axis,stage):
    """V6 small forward-sector reacquisition + independent 3-step ridge persistence test."""
    if len(axis)<2:return None
    recent=unit(tangent(axis[max(0,len(axis)-6)][0],axis[max(0,len(axis)-6)][1],axis[-1][0],axis[-1][1],y))
    if not np.isfinite(recent).all():recent=t
    if cosangle(recent,t)<0:recent=-recent
    base=unit(.9*(.55*t+.45*recent)+.1*g)
    if not np.isfinite(base).all():base=t
    out=None;bestscore=-np.inf
    for fwd in (12,18,24,30,36,48):
        dx,dy=km2deg(fwd*base[0],fwd*base[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):continue
        off,xx,yy,z=F.profile(xp,yp,[-base[1],base[0]],84,4)
        good=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['a'])&np.isfinite(z['gx'])&np.isfinite(z['gy'])&(z['g']>0)
        cand=ridge_band(z['g'],good,4,.76,.45,20)
        for j in cand:
            x1,y1=xx[j],yy[j];gg=z['g'][j];gc=z['gc'][j];a=z['a'][j]
            tc=unit([-z['gy'][j],z['gx'][j]])
            if not np.isfinite(tc).all():continue
            jump=float(geo(x,y,x1,y1));mv=unit(tangent(x,y,x1,y1,(y+y1)/2))
            if jump<7.8 or jump>65 or not np.isfinite(mv).all():continue
            turn=ang(t,tc);move_turn=ang(base,mv)
            if turn>125 or move_turn>80:continue
            gd=cosangle(tc,g)
            if x<124 and (tc[1]<-.05 or gd<-.45):continue
            if 124<=x<130 and gd<-.60:continue
            if x>=130 and (tc[0]<-.60 or gd<-.65):continue
            if 128.5<=x<=132.5:
                if tc[0]<-.45:continue
                if len(axis)>8:
                    zold=axis[-8];dxnet=(x1-zold[0])*KMLAT*np.cos(np.deg2rad((y1+zold[1])/2))
                    if dxnet<-25:continue
            if stage==0 and 128.7<=x<=131.7:
                if y1>=30.45 and x1<130.25:continue
                if y>=29.7 and tc[0]<-.02:continue
                if y1>=29.7 and x1<130.05 and tc[0]<.18:continue
            if 122<=x<=126.2 and 24.4<=y<=26.9 and len(axis)>7:
                old=axis[-7];chord=geo(old[0],old[1],x1,y1)
                seg=np.asarray(axis[-7:]+[[x1,y1]])
                plen=float(np.sum(geo(seg[:-1,0],seg[:-1,1],seg[1:,0],seg[1:,1])))
                eff=chord/max(plen,1e-10);dxnet=(x1-old[0])*KMLAT*np.cos(np.deg2rad((y1+old[1])/2))
                if eff<.48 or dxnet<-12 or turn>68:continue
            gr=gg/F.g95;gcr=gc/F.gc95
            if gr<.18 or gcr<.18:continue
            dhard=min(.18,.12+.0015*max(0,jump-12));da=abs(a-a0)
            if da>dhard:continue
            lp=0.
            if len(axis)>18:
                old=np.asarray(axis[:-12]);dm=float(np.min(geo(x1,y1,old[:,0],old[:,1])))
                if dm<24:continue
                if dm<45:lp=(45-dm)/21
            closed,known=F.topo.query(x1,y1,a,18) if F.topo and x>=126 else (False,False)
            tp=1.1 if known and closed else (.1 if not known and F.topo and x>=126 else 0)
            supp=support_ahead(F,x1,y1,tc)
            if supp<.18:continue
            mini,dxmini=mini_track(F,x1,y1,tc)
            if mini<.20:continue
            if 128.5<=x<=132.5 and dxmini<-5:continue
            sr=.45*supp+.55*mini
            sc=min(gr,1.7)+.7*min(gcr,1.7)+.55*min(sr,1.5)+.1*max(gd,0)
            sc-=.22*(off[j]/84)**2+.55*(turn/90)**2+.35*(move_turn/70)**2
            sc-=.55*(da/.12)**2+.20*((jump-12)/55)**2+.30*(max(0,.1-gd)/1.1)**2+1.6*lp+tp
            if sc>bestscore:bestscore=sc;out=(float(x1),float(y1),float(gg),float(a),float(off[j]),float(turn),tc,float(gd),float(jump))
    return out

def regularize_ridge(F,xy,recovery):
    """MATLAB V6 2-pass Jacobi ridge-constrained corner regularization, raw path preserved."""
    xx=np.asarray(xy,float).copy();ref=xx.copy();changed=np.zeros(len(xx),bool);n=len(xx)
    if n<9:return xx,changed
    for _ in range(2):
        turn=turning(xx[:,0],xx[:,1]);flag=np.zeros(n,bool)
        for i in range(3,n-3):
            if not np.isfinite(turn[i]):continue
            local=turn[max(1,i-2):min(n-1,i+3)];local=local[np.isfinite(local)]
            med=np.median(local) if len(local) else 0
            if turn[i]>=78 or (turn[i]>=30 and turn[i]>=med+6):flag[max(3,i-2):min(n-3,i+3)]=True
        for i in np.flatnonzero(recovery):flag[max(3,i-3):min(n-3,i+4)]=True
        if not flag.any():break
        old=xx.copy()
        for i in np.flatnonzero(flag):
            x,y=old[i]
            t=unit(tangent(*old[i-2],*old[i+2],y))
            if not np.isfinite(t).all():continue
            prev=tangent(x,y,*old[i-1],(y+old[i-1,1])/2)
            nxt=tangent(x,y,*old[i+1],(y+old[i+1,1])/2)
            delta=.62*.5*(prev+nxt)
            dx,dy=km2deg(delta[0],delta[1],y);xt=x+dx;yt=y+dy
            off,xc,yc,z=F.profile(xt,yt,[-t[1],t[0]],18,1.5,('g','gc','a'))
            good=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['a'])&(z['g']>0)&(z['gc']>0)
            if not good.any():continue
            graw=F.get('g',y,x);gcraw=F.get('gc',y,x)
            if not np.isfinite(graw) or graw<=0:graw=.35*F.g95
            if not np.isfinite(gcraw) or gcraw<=0:gcraw=.35*F.gc95
            aa=F.get('a',old[i-1,1],old[i-1,0]);bb=F.get('a',old[i+1,1],old[i+1,0]);oldturn=turn3(*old[i-1],*old[i],*old[i+1])
            if not np.isfinite(oldturn):oldturn=180
            bestscore=-np.inf;best=None
            for j in np.flatnonzero(good):
                move=geo(ref[i,0],ref[i,1],xc[j],yc[j])
                if move>14 or z['g'][j]<.68*graw or z['gc'][j]<.55*gcraw:continue
                da=max((abs(z['a'][j]-aa) if np.isfinite(aa) else 0),(abs(z['a'][j]-bb) if np.isfinite(bb) else 0))
                if da>.1:continue
                newturn=turn3(*old[i-1],xc[j],yc[j],*old[i+1])
                if not np.isfinite(newturn):continue
                if oldturn-newturn<1.5 and oldturn<78:continue
                if newturn>oldturn+1:continue
                targetdist=geo(xt,yt,xc[j],yc[j])
                sc=min(z['g'][j]/F.g95,1.8)+.55*min(z['gc'][j]/F.gc95,1.8)
                sc-=.42*(targetdist/18)**2+.75*(newturn/90)**2+.35*(da/.1)**2+.2*(move/14)**2
                if sc>bestscore:bestscore=sc;best=(xc[j],yc[j],newturn)
            if best is not None and best[2]<=oldturn:
                xx[i,:2]=best[:2]
                if geo(*old[i],*xx[i])>.25:changed[i]=True
    return xx,changed

def track_adt_v6(lon,lat,adt,config=V6()):
    """Return (axis[N,2], metadata). Axis is surface ADT gradient-front proxy, not 50m velocity core."""
    F=Fields(lon,lat,adt,build_topo=config.use_topology)
    seed=taiwan_seed(F)
    axis=seed.tolist();recs=[False]*len(axis)
    t=unit(tangent(*axis[max(0,len(axis)-4)],*axis[-1],axis[-1][1]))
    if not np.isfinite(t).all():
        gx=F.get('gx',axis[-1][1],axis[-1][0]);gy=F.get('gy',axis[-1][1],axis[-1][0]);t=unit([-gy,gx])
    if not np.isfinite(t).all():raise RuntimeError('ADT V6 seed tangent undefined')
    if t[1]<0:t=-t
    stage=0;fails=0;width=48.;reason='maximum point number reached'
    for _ in range(config.max_points-len(axis)):
        x,y=axis[-1];a0=F.get('a',y,x)
        while stage<3 and gate_pass(x,y,GATES[stage],[.35,.35,.55][stage]):stage+=1
        g=guide(x,y,stage);blend=.18 if stage>=3 else .12
        tpred=unit((1-blend)*t+blend*g)
        dx,dy=km2deg(12*tpred[0],12*tpred[1],y);xp=x+dx;yp=y+dy
        if not F.inside(xp,yp):reason='predicted point left domain';break
        off,xx,yy,z=F.profile(xp,yp,[-tpred[1],tpred[0]],width,3)
        good=np.isfinite(z['g'])&np.isfinite(z['gc'])&np.isfinite(z['a'])&np.isfinite(z['gx'])&np.isfinite(z['gy'])&(z['g']>0)
        cand=ridge_band(z['g'],good,3,.8,.55,15)
        if len(cand)>18:
            merit=z['g'][cand]/F.g95+.45*z['gc'][cand]/F.gc95-.08*np.abs(off[cand])/width
            cand=cand[np.argsort(-merit)[:18]]
        best=None;bs=-np.inf
        for j in cand:
            xc,yc=float(xx[j]),float(yy[j]);gc=float(z['g'][j]);gcc=float(z['gc'][j]);ac=float(z['a'][j])
            tc=unit([-z['gy'][j],z['gx'][j]])
            if not np.isfinite(tc).all():continue
            turn=ang(t,tc)
            if turn>85:continue
            gd=cosangle(tc,g)
            if x<124 and (tc[1]<.05 or gd<-.25):continue
            if 124<=x<130 and gd<-.45:continue
            if x>=130 and (tc[0]<-.45 or gd<-.5):continue
            tok=tokexit=tweff=twret=twturn=twtopo=0.
            if 128.5<=x<=132.5:
                if tc[0]<-.45:continue
                if tc[0]<-.18:tok=((-.18-tc[0])/.27)**2
                if len(axis)>8:
                    old=axis[-8];d=(xc-old[0])*KMLAT*np.cos(np.deg2rad((yc+old[1])/2))
                    if d< -5:tok+=min(2.,(-5-d)/35)
            if stage==0 and 128.7<=x<=131.7:
                if yc>=30.45 and xc<130.25:continue
                if yc>=29.7 and xc<130.05:tokexit+=((130.05-xc)/.55)**2+((yc-29.7)/.9)**2
                if y>=29.7:
                    if tc[0]<-.02:continue
                    if tc[0]<.18:tokexit+=((.18-tc[0])/.20)**2
            if 122<=x<=126.2 and 24.4<=y<=26.9 and len(axis)>7:
                old=axis[-7];ch=geo(old[0],old[1],xc,yc)
                seg=np.asarray(axis[-7:]+[[xc,yc]])
                plen=np.sum(geo(seg[:-1,0],seg[:-1,1],seg[1:,0],seg[1:,1]))
                ef=ch/max(plen,1e-10)
                if ef<.48:continue
                if ef<.72:tweff=((.72-ef)/.24)**2
                dw=(xc-old[0])*KMLAT*np.cos(np.deg2rad((yc+old[1])/2))
                if dw<-12:continue
                if dw<-2:twret=((-2-dw)/10)**2
                if turn>68:continue
                if turn>42:twturn=((turn-42)/26)**2
                if F.topo:
                    closed,known=F.topo.query(xc,yc,ac)
                    twtopo=1.35 if closed and known else (.05 if not known else 0)
            da=abs(ac-a0);relaxed=False;supp=0
            if da>.11:
                if da>.175 or gc/F.g95<.45 or gcc/F.gc95<.35:continue
                supp=support_multi(F,xc,yc,tc)
                if supp<.38:continue
                relaxed=True
            closed,known=F.topo.query(xc,yc,ac) if F.topo and x>=126 else (False,False)
            tp=1.15 if closed and known else (.1 if F.topo and x>=126 and not known else 0)
            loop=0.
            if len(axis)>15:
                old=np.asarray(axis[:-10]);dmin=float(np.min(geo(xc,yc,old[:,0],old[:,1])))
                if dmin<38:loop=1+(38-dmin)/38
            prog=0.
            if x>=128.5 and len(axis)>10:
                old=axis[-9];dnet=(xc-old[0])*KMLAT*np.cos(np.deg2rad((yc+old[1])/2))
                if dnet<-5:prog=min(2,(-5-dnet)/45)
            dx2,dy2=km2deg(12*tc[0],12*tc[1],yc)
            cc=F.get('gc',yc+dy2,xc+dx2)
            look=min(cc/F.gc95,1.5) if np.isfinite(cc) else 0.
            if relaxed:look=max(look,min(supp,1.5))
            sc=min(gc/F.g95,1.5)+.45*min(gcc/F.gc95,1.5)+.25*look
            sc-=.25*(off[j]/width)**2+.65*(turn/60)**2
            sc-=.8*(da/(.11 if relaxed else .05))**2+.35*(max(0,.15-gd)/1.15)**2
            sc-=2.8*loop+1.2*prog+1.35*tok+2.4*tokexit+1.1*tweff+1.15*twret+1.2*twturn+tp+twtopo
            sc+=.08*max(gd,0)
            if sc>bs:bs=sc;best=(xc,yc,gc,ac,float(off[j]),turn,tc,gd,np.nan)
        used=False
        if best is None:
            fails+=1
            if config.enable_recovery and fails>=2:
                best=recover(F,x,y,a0,t,g,axis,stage)
                if best is not None:used=True
            if best is None:
                if fails>=6:reason='six local searches and recovery failed';break
                width=min(width+4,60.);continue
        fails=0;width=48.
        xc,yc,gc,ac,off,turn,tc,gd,jump=best
        axis.append([xc,yc]);recs.append(used)
        t=unit((1-(.50 if used else .30))*t+(.50 if used else .30)*tc)
        if xc>=config.stop_lon and yc>=config.stop_lat and stage>=3:
            reason='eastern downstream target reached';break
    raw=np.asarray(axis,float);rec=np.asarray(recs,bool)
    if config.regularize and len(raw)>=9:final,changed=regularize_ridge(F,raw,rec)
    else:final=raw.copy();changed=np.zeros(len(raw),bool)
    meta={'stop_reason':reason,'success':bool(reason=='eastern downstream target reached'),
          'n_points':int(len(raw)),'last_lon':float(raw[-1,0]),'last_lat':float(raw[-1,1]),
          'n_recovery':int(rec.sum()),'n_regularized':int(changed.sum()),
          'ridge_g95':float(F.g95),'ridge_gc95':float(F.gc95),'method':'ADT_V6_PythonPort'}
    return final,meta,raw,rec,changed


def read_adt_netcdf(path,expected_date=None,lon_min=120.,lon_max=150.,lat_min=20.,lat_max=40.):
    """Lazily subset each daily NetCDF to ADT study window before computing, verifying the actual time."""
    with xr.open_dataset(path,decode_times=True,mask_and_scale=True) as ds:
        names=ds.variables
        lonkey=next((k for k in ('longitude','lon') if k in names),None)
        latkey=next((k for k in ('latitude','lat') if k in names),None)
        adtkey=next((k for k in ('adt','ADT') if k in names),None)
        if not (lonkey and latkey and adtkey):raise KeyError('Expected adt, longitude, latitude; got '+str(list(names)))
        z=ds[adtkey]
        for key,lo,hi in ((lonkey,lon_min,lon_max),(latkey,lat_min,lat_max)):
            coords=ds[key].values
            if coords.ndim!=1:raise ValueError('Only 1D longitude/latitude supported')
            sel=slice(lo,hi) if coords[0]<coords[-1] else slice(hi,lo)
            z=z.sel({key:sel})
        other=[k for k in z.dims if k not in (lonkey,latkey)]
        for k in other:
            if z.sizes[k]==1: z=z.isel({k:0});continue
            if expected_date is not None and k in ds.coords:
                try:
                    dt=pd.to_datetime(ds[k].values)
                    target=pd.Timestamp(expected_date)
                    dif=np.abs(dt-target)
                    ix=int(dif.argmin())
                    if dif[ix]>pd.Timedelta(hours=18):raise ValueError('ADT time mismatch')
                    z=z.isel({k:ix});continue
                except Exception as ex:raise ValueError(f'Cannot confirm time coordinate {k}: {ex}') from ex
            raise ValueError(f'Unexpected non-singleton dimension {k}={z.sizes[k]}')
        if expected_date is not None and 'time' in ds.coords and ds['time'].size==1:
            # Do not silently trust a stale name if time coordinate contradicts it.
            tc=pd.to_datetime(np.asarray(ds['time'].values).reshape(-1)[0]);assert abs(tc-pd.Timestamp(expected_date))<=pd.Timedelta(hours=18),f'time mismatch {tc} vs {expected_date}'
        z=z.transpose(latkey,lonkey)
        lon=np.asarray(z[lonkey].values,float);lat=np.asarray(z[latkey].values,float);a=np.asarray(z.values,float)
        if lon[0]>lon[-1]:lon=lon[::-1];a=a[:,::-1]
        if lat[0]>lat[-1]:lat=lat[::-1];a=a[::-1]
        a[np.abs(a)>100]=np.nan
        if min(len(lon),len(lat))<10 or np.isfinite(a).mean()<.3:raise ValueError('ADT subset too small or mostly invalid')
        return lon,lat,a


In [ ]:
# ===== Cell 2：读入1993—2023每日ADT文件清单，审计重复/缺失/日期版本 =====
from datetime import date
pat=re.compile(r'(?<!\d)((?:19|20)\d{6})(?!\d)')
records=[]
for p in sorted(ADT_ROOT.glob('????/*.nc')):
    ds=pat.findall(p.name)
    if not ds:continue
    try:d=pd.to_datetime(ds[0],format='%Y%m%d')
    except ValueError:continue
    if not (pd.Timestamp(cfg.START_DATE)<=d<=pd.Timestamp(cfg.END_DATE)):continue
    if p.parent.name!=str(d.year):continue
    records.append({'date':str(d.date()),'path':str(p),'bytes':p.stat().st_size,'revision_token':ds[1] if len(ds)>1 else ''})
manifest=pd.DataFrame(records,columns=['date','path','bytes','revision_token'])
if manifest.empty:raise RuntimeError('没有扫描到匹配的ADT文件，请检查年份目录与文件命名')
dup=manifest[manifest.duplicated('date',keep=False)].copy()
manifest=manifest.sort_values(['date','bytes'],ascending=[True,False]).drop_duplicates('date').reset_index(drop=True)
expected=pd.date_range(cfg.START_DATE,cfg.END_DATE)
missing_dates=sorted(set(expected.date)-set(pd.to_datetime(manifest.date).dt.date))
manifest.to_csv(REP/'01_ADT原始日期Manifest.csv',index=False,encoding='utf-8-sig')
pd.DataFrame({'missing_date':list(map(str,missing_dates))}).to_csv(REP/'02_ADT缺失日期.csv',index=False,encoding='utf-8-sig')
dup.to_csv(REP/'03_ADT同日期多文件.csv',index=False,encoding='utf-8-sig')
coverage=manifest.assign(year=pd.to_datetime(manifest.date).dt.year).groupby('year').size().rename('n_files').reset_index()
coverage.to_csv(REP/'04_ADT年覆盖统计.csv',index=False,encoding='utf-8-sig')
print('ADT文件:',len(manifest),'理论自然日:',len(expected),'缺失日期:',len(missing_dates),'重复记录:',len(dup))
display(coverage)
# 绝不把文件名中的第二个 YYYYMMDD（如20241015：产品版本/生成标签）识别成观测日期。
assert manifest.date.is_unique


In [ ]:
# ===== Cell 3：真实NetCDF文件元数据抽查＋追踪器合成流轴冒烟测试 =====
from scipy.spatial import cKDTree
first=manifest.iloc[0]
lon,lat,adt=read_adt_netcdf(first.path,first.date)
print('NetCDF实测:',first.date,'ROI shape:',adt.shape,'lon:',lon[[0,-1]],'lat:',lat[[0,-1]],'finite fraction',np.isfinite(adt).mean())
assert adt.shape==(len(lat),len(lon))
# 人工构造具有台湾-东海-南日本-黑潮延伸体形态的连续ADT梯度锋面，核对全段能通过V6追踪
lat0=np.arange(20,40.001,.125);lon0=np.arange(120,150.001,.125)
x0=np.r_[np.linspace(122,124,45),np.linspace(124,131,65),np.linspace(131,145,150)]
y0=np.r_[np.linspace(22.2,26.5,45),np.linspace(26.5,29.2,65),29.2+2.8*(1-np.exp(-(x0[110:]-131)/8))+.7*np.sin((x0[110:]-131)/3)]
p0=np.c_[x0,y0];LON,LAT=np.meshgrid(lon0,lat0);cos=np.cos(np.deg2rad(31))
idx=cKDTree(np.c_[p0[:,0]*cos,p0[:,1]]).query(np.c_[LON.ravel()*cos,LAT.ravel()])[1]
nearest=p0[idx];dirs=np.gradient(p0,axis=0)[idx]
sg=((LON.ravel()-nearest[:,0])*dirs[:,1]-(LAT.ravel()-nearest[:,1])*dirs[:,0]/cos)/np.maximum(np.linalg.norm(dirs,axis=1),1e-9)
synthetic=.6*np.tanh(sg.reshape(LON.shape)/.32)
test_axis,test_meta,_,_,_=track_adt_v6(lon0,lat0,synthetic,V6())
print('[SYNTHETIC V6 TRACK]',test_meta)
assert test_meta['success'],'纯Python V6追踪器未通过合成真值测试，不得继续全量处理'
# 测试真实ADT字段的关键种子，不强制要求当天必须追踪成功；不通过会按原版V6记录失败等待人工复核
try:
    a,m,_,_,_=track_adt_v6(lon,lat,adt,V6(use_topology=cfg.USE_TOPOLOGY,regularize=cfg.RIDGE_REGULARIZATION))
    print('[REAL ADT DAY 1]',m)
except Exception as e:print('[REAL ADT DAY 1] tracker failed (will log, not discard silently):',type(e).__name__,str(e))
del adt,synthetic,LON,LAT;gc.collect()


In [ ]:
# ===== Cell 4：自动真实日期smoke → 全量断点续跑 → 输出Python NPZ流轴、元数据与失败日志 =====
from collections import Counter
files={str(x.date):x.path for x in manifest.itertuples(index=False)}
known_adt_dates=sorted(files)
smoke=[]
for d in cfg.SMOKE_DATES:
    if d in files:smoke.append(d)
    else:
        closest=min(known_adt_dates,key=lambda x:abs((pd.Timestamp(x)-pd.Timestamp(d)).days))
        smoke.append(closest)
smoke=list(dict.fromkeys(smoke))

def run_one_adt(day):
    stem=pd.Timestamp(day).strftime('%Y%m%d')
    output=ADT_AX/(stem+'.npz');metap=META/(stem+'.json')
    if output.exists() and not cfg.FORCE_RETRACK:return 'existing_success'
    if metap.exists() and not cfg.RETRY_FAILED and not cfg.FORCE_RETRACK:
        try:
            old=json.loads(metap.read_text(encoding='utf-8'))
            if old.get('status') in ('failed','qc_rejected'):return 'existing_failed'
        except Exception:pass
    t0=time.perf_counter()
    try:
        lo,la,aa=read_adt_netcdf(files[day],day)
        xy,meta,raw,recovery,reg=track_adt_v6(lo,la,aa,V6(use_topology=cfg.USE_TOPOLOGY,regularize=cfg.RIDGE_REGULARIZATION))
        meta.update({'date':day,'source_nc':files[day],'walltime_s':round(time.perf_counter()-t0,3)})
        # 禁止将短轴误当有效完整黑潮流轴，保留原因供人工复核。
        passed=meta['success'] and len(xy)>=20 and np.max(xy[:,0])>=cfg.MIN_TERMINAL_LON
        meta['status']='success' if passed else 'qc_rejected'
        meta['qc_warning']=bool((meta['n_recovery']>20) or (meta['n_regularized']>30))
        if passed:
            np.savez_compressed(output,axis_lon=xy[:,0].astype('float32'),axis_lat=xy[:,1].astype('float32'),
                 axis_lon_raw=raw[:,0].astype('float32'),axis_lat_raw=raw[:,1].astype('float32'),
                 axis_recovery=recovery.astype('uint8'),axis_regularized=reg.astype('uint8'),
                 date=np.array(day),method=np.array('ADT_V6_PythonPort'))
    except Exception as exc:
        meta={'date':day,'status':'failed','source_nc':files[day],
              'walltime_s':round(time.perf_counter()-t0,3),'error':repr(exc),'traceback':traceback.format_exc(limit=6)}
    metap.write_text(json.dumps(meta,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    return meta['status']

def collect_adt_tracking_report():
    rows=[]
    for p in META.glob('*.json'):
        try:
            d=json.loads(p.read_text(encoding='utf-8'))
            rows.append({k:v for k,v in d.items() if k!='traceback'})
        except Exception:continue
    df=pd.DataFrame(rows)
    if not df.empty:df=df.sort_values('date')
    df.to_csv(REP/'05_ADT逐日追踪及QC报告.csv',index=False,encoding='utf-8-sig')
    return df

print('Running six-date V6 smoke:',smoke)
for day in smoke:
    status=run_one_adt(day)
    print('  ',day,status)
smoke_report=collect_adt_tracking_report()
if cfg.RUN_FULL:
    tstart=time.perf_counter()
    counter=Counter()
    for i,day in enumerate(tqdm(known_adt_dates,desc='1993–2023 Python ADT V6',unit='day'),1):
        status=run_one_adt(day);counter[status]+=1
        if i%cfg.PRINT_EVERY==0:print('[ADT V6]',i,'/',len(known_adt_dates),'status',dict(counter),'elapsed_hours',round((time.perf_counter()-tstart)/3600,2),flush=True)
        if i%cfg.ZIP_EVERY_N_DAYS==0:collect_adt_tracking_report()
    print('ALL ADT AVAILABLE DATES PROCESSED:',len(known_adt_dates),dict(counter))
else:print('RUN_FULL=False: only smoke was processed; this is not the full validation.')
adt_tracking_report=collect_adt_tracking_report()
print('All accepted ADT axes:',len(list(ADT_AX.glob('*.npz'))),'status:',adt_tracking_report.status.value_counts(dropna=False).to_dict())
# 追踪失败日不参与误差评价；但是失败率会如实报告，不能选择性删除。


## 第二阶段：内置独立验证（不依赖外部 Python 模块）

以下代码包括轨迹加密采样、地球表面近似大圆距离、六断面线段真交点、成对分块 Bootstrap、月尺度趋势及英文／中文绘图。预测部分只读取带日期的真实模型输出。

In [ ]:
"""R7.0 ADT V6 independent observational axis validation.
Requires scipy, numpy, pandas, matplotlib. No forecast model fitting and no ADT leakage.
"""
from __future__ import annotations
import os, re, json, csv, math, hashlib, zipfile, warnings, traceback
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr

SECTIONS={
 'A':((121.50,23.50),(125.00,22.75)),
 'B':((121.50,26.25),(124.25,24.25)),
 'PN':((126.00,29.00),(128.15,27.30)),
 'C':((130.25,31.25),(132.50,27.00)),
 'D':((137.00,34.75),(138.00,27.75)),
 'E':((141.75,40.00),(145.00,31.50))}

@dataclass
class Settings:
    ROOT:str='/mnt/d/KuroshioDaily1993_2025'
    ADT_ROOT:str='/mnt/f/01-CMEMS/SEALEVEL_GLO_PHY_L4_MY_008_047/cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D'
    START_DATE:str='1993-01-01'
    END_DATE:str='2023-12-31'
    YEARS:tuple=tuple(range(1993,2024))
    GLORYS_TRACK_REL:str='final/tracks_final'
    ADT_AXIS_REL:str='R70_ADT_Independent/adt_axes_mat'
    OUT_REL:str='R70_ADT_Independent/results'
    PRED_MANIFEST_REL:str='R70_ADT_Independent/predictions_manifest.csv'
    MIN_ADT_POINTS:int=20
    MIN_GLO_POINTS:int=20
    REQUIRE_ADT_TERMINAL:bool=True
    MIN_TERMINAL_LON:float=143.0
    MIN_MONTH_DAYS:int=15
    DENSIFY_KM:float=5.0
    BOOT_N:int=400
    BLOCK_DAYS:int=30
    MONTHLY_BOOT_BLOCK:int=12
    FULL_RUN:bool=False
    SMOKE_N:int=18
    FIGURE_SAMPLE_N:int=8
    RANDOM_SEED:int=20261015

def _d(x):return pd.Timestamp(x).normalize()
def ensure(o):
    for k in ('reports','figures_en','figures_cn','tables','daily','monthly'):(Path(o)/k).mkdir(parents=True,exist_ok=True)

def scan_adt_raw(root,start,end):
    """Scan YYYY subfolders; use FIRST YYYYMMDD token (later 20241015 is product version)."""
    root=Path(root);out=[];start=_d(start);end=_d(end)
    if not root.is_dir():raise FileNotFoundError(f'ADT source missing: {root}')
    pat=re.compile(r'(?<!\d)(19\d{6}|20\d{6})(?!\d)')
    for p in sorted(root.glob('????/*.nc')):
        matches=pat.findall(p.name)
        if not matches:continue
        try:t=pd.to_datetime(matches[0],format='%Y%m%d')
        except ValueError:continue
        if not start<=t<=end:continue
        if p.parent.name!=str(t.year):continue
        out.append({'date':str(t.date()),'file':str(p),'bytes':p.stat().st_size,'filename_date':True})
    df=pd.DataFrame(out,columns=['date','file','bytes','filename_date'])
    if not df.empty:df=df.sort_values(['date','bytes'],ascending=[True,False]).drop_duplicates('date')
    return df

def load_axis(path,kind):
    path=Path(path)
    if kind=='adt':
        if path.suffix.lower()=='.npz':
            with np.load(path,allow_pickle=False) as z:
                lon=np.asarray(z['axis_lon'],float).reshape(-1);lat=np.asarray(z['axis_lat'],float).reshape(-1)
        else:
            z=loadmat(path,variable_names=['axis_lon','axis_lat','axis_lon_raw','axis_lat_raw','axis_recovery','axis_regularized'],squeeze_me=True)
            lon=np.asarray(z['axis_lon'],float).reshape(-1);lat=np.asarray(z['axis_lat'],float).reshape(-1)
    elif kind=='glo':
        with np.load(path,allow_pickle=False) as z:
            lon=np.asarray(z['lon'],float).reshape(-1);lat=np.asarray(z['lat'],float).reshape(-1)
    elif kind=='pred':
        with np.load(path,allow_pickle=False) as z:
            if all(k in z.files for k in ('branch1_lon','branch1_lat','branch2_lon','branch2_lat')):
                a=np.stack([z['branch1_lon'].reshape(-1),z['branch1_lat'].reshape(-1)],axis=1)
                b=np.stack([z['branch2_lon'].reshape(-1),z['branch2_lat'].reshape(-1)],axis=1)
                return [a,b]
            lon=np.asarray(z['lon'],float).reshape(-1);lat=np.asarray(z['lat'],float).reshape(-1)
    else: raise ValueError(kind)
    xy=np.stack([lon,lat],axis=1)
    good=np.isfinite(xy).all(axis=1)&(xy[:,0]>=110)&(xy[:,0]<=160)&(xy[:,1]>=10)&(xy[:,1]<=48)
    # Reject internal NaN gaps instead of connecting across disconnected pieces.
    if not good.all():
        segments=[]; start=None
        for i,ok in enumerate(good):
            if ok and start is None:start=i
            if start is not None and (not ok or i==len(good)-1):
                j=i if not ok else i+1
                if j-start>=2:segments.append(xy[start:j])
                start=None
        if not segments: return np.empty((0,2))
        return max(segments,key=len)
    return xy

def km_xy(xy,lat0=31.0):
    a=np.asarray(xy,float)
    return np.stack([a[...,0]*(111.195*np.cos(np.deg2rad(lat0))),a[...,1]*111.195],axis=-1)

EARTH_RADIUS_KM=6371.0088

def sphere_xyz(xy):
    """Earth-centered Cartesian km for locally near-geodesic point distance."""
    a=np.deg2rad(np.asarray(xy,float))
    lon=a[...,0];lat=a[...,1];co=np.cos(lat)
    return EARTH_RADIUS_KM*np.stack([co*np.cos(lon),co*np.sin(lon),np.sin(lat)],axis=-1)

def chord_to_gc_km(chord):
    return 2*EARTH_RADIUS_KM*np.arcsin(np.clip(np.asarray(chord)/(2*EARTH_RADIUS_KM),0,1))

def seg_lengths(xy):
    a=sphere_xyz(xy)
    return chord_to_gc_km(np.linalg.norm(np.diff(a,axis=0),axis=1))

def densify(xy,step_km=5.0):
    xy=np.asarray(xy,float)
    if len(xy)<2:return xy.copy()
    pieces=[xy[0]]
    for a,b,d in zip(xy[:-1],xy[1:],seg_lengths(xy)):
        # Never silently connect a genuine gross discontinuity.
        if d>250:continue
        n=max(1,int(np.ceil(d/step_km)))
        pieces.extend(a+(b-a)*(k/n) for k in range(1,n+1))
    return np.asarray(pieces)

def polyline_rmse(a,b,step=5.,lat0=31.):
    """Symmetric nearest-point distance, densely sampled polylines; not directed Hausdorff."""
    aa=densify(a,step);bb=densify(b,step)
    if min(len(aa),len(bb))<3:return {'rmse_km':np.nan,'mae_km':np.nan,'p90_km':np.nan,'bias_AtoB_km':np.nan,'bias_BtoA_km':np.nan}
    ax,bx=sphere_xyz(aa),sphere_xyz(bb)
    ab=chord_to_gc_km(cKDTree(bx).query(ax)[0]);ba=chord_to_gc_km(cKDTree(ax).query(bx)[0])
    merged=np.r_[ab,ba]
    return {'rmse_km':float(np.sqrt(np.mean(merged**2))),'mae_km':float(np.mean(merged)),
            'p90_km':float(np.quantile(merged,.9)),'bias_AtoB_km':float(ab.mean()),'bias_BtoA_km':float(ba.mean())}

def _cross(a,b):return float(a[0]*b[1]-a[1]*b[0])

def intersections(axis,section):
    """All TRUE polyline–section intersections (lon, lat, position_km, order)"""
    s1,s2=section
    sy=km_xy([s1,s2]); r=sy[1]-sy[0]
    if len(axis)<2:return []
    pxy=km_xy(axis)
    out=[]
    for i in range(len(axis)-1):
        p=pxy[i];s=pxy[i+1]-p;den=_cross(s,r)
        if abs(den)<1e-10:continue
        d=sy[0]-p
        t=_cross(d,r)/den; u=_cross(d,s)/den
        if -1e-9<=t<=1+1e-9 and -1e-9<=u<=1+1e-9:
            q=axis[i]+np.clip(t,0,1)*(axis[i+1]-axis[i])
            val=float(chord_to_gc_km(np.linalg.norm(sphere_xyz([q])[0]-sphere_xyz([s1])[0])))
            out.append((float(q[0]),float(q[1]),val,i+float(t)))
    out.sort(key=lambda t:t[3])
    uniq=[]
    for q in out:
        if not any(abs(q[2]-p[2])<.05 for p in uniq):uniq.append(q)
    return uniq

def section_records(axis,date,source,extra=None):
    rows=[]
    for key,segment in SECTIONS.items():
        crossings=intersections(axis,segment)
        # Pre-registered choice: FIRST crossing encountered along Taiwan->KE axis.
        v=crossings[0] if crossings else (np.nan,np.nan,np.nan,np.nan)
        rows.append({'date':date,'source':source,'section':key,'lon':v[0],'lat':v[1],
                     'distance_km':v[2],'n_intersections':len(crossings),
                     'ambiguous':len(crossings)>1,**(extra or {})})
    return rows

def _numeric_date_idx(dates):return pd.to_datetime(dates).to_numpy(dtype='datetime64[D]')

def paired_boot_ci(x,block=30,nboot=400,seed=1):
    x=np.asarray(x,float);x=x[np.isfinite(x)];n=len(x)
    if n<5:return (np.nan,np.nan)
    rng=np.random.default_rng(seed);b=max(1,min(block,n));samples=[]
    for _ in range(nboot):
        ids=[]
        while len(ids)<n:
            start=int(rng.integers(0,n));ids.extend((start+np.arange(b))%n)
        samples.append(np.mean(x[np.asarray(ids[:n],dtype=int)]))
    return tuple(np.percentile(samples,[2.5,97.5]).tolist())

def _ols(t,y):
    t=np.asarray(t,float);y=np.asarray(y,float)
    M=np.column_stack([np.ones(len(t)),t]);coef=np.linalg.lstsq(M,y,rcond=None)[0]
    return float(coef[1]),M@coef

def trend_paired_boot(t,adt,glo,block=12,nboot=500,seed=1):
    """Paired residual circular block bootstrap: preserves each fitted trend and cross-product residual pairing."""
    t=np.asarray(t,float);adt=np.asarray(adt,float);glo=np.asarray(glo,float)
    if len(t)<24:return {'adt_km_decade':np.nan,'glorys_km_decade':np.nan,'difference_km_decade':np.nan,'diff_ci95_lo':np.nan,'diff_ci95_hi':np.nan}
    a,af=_ols(t,adt);g,gf=_ols(t,glo)
    resid=np.column_stack([adt-af,glo-gf]);n=len(t);rng=np.random.default_rng(seed); b=min(block,n)
    vals=[]
    for _ in range(nboot):
        ii=[]
        while len(ii)<n:
            st=int(rng.integers(n));ii.extend((st+np.arange(b))%n)
        r=resid[np.asarray(ii[:n],int)]
        sa,_=_ols(t,af+r[:,0]);sg,_=_ols(t,gf+r[:,1]);vals.append((sg-sa)*10)
    q=np.quantile(vals,[.025,.975])
    return {'adt_km_decade':a*10,'glorys_km_decade':g*10,'difference_km_decade':(g-a)*10,
            'diff_ci95_lo':float(q[0]),'diff_ci95_hi':float(q[1])}

def paired_series_daily(adt_dir,glo_dir,start,end,smoke_n=0,terminal=143.0):
    a_dir=Path(adt_dir);g_dir=Path(glo_dir)
    pa={p.stem:p for p in a_dir.glob('*.npz') if re.fullmatch(r'\d{8}',p.stem)}
    pg={p.stem:p for p in g_dir.glob('*.npz') if re.fullmatch(r'\d{8}',p.stem)}
    start=pd.Timestamp(start).strftime('%Y%m%d');end=pd.Timestamp(end).strftime('%Y%m%d')
    dates=sorted(k for k in pa.keys()&pg.keys() if start<=k<=end)
    if smoke_n>0 and len(dates)>smoke_n:
        dates=sorted({dates[int(j)] for j in np.linspace(0,len(dates)-1,smoke_n,dtype=int)})
    return [(k,pa[k],pg[k]) for k in dates],pa,pg

def process_pairs(pairs,min_adt=20,min_glo=20,require_terminal=True,terminal=143.,step=5.,out=None):
    daily=[];sections=[];failure=[];paths=[]
    for j,(dt,ap,gp) in enumerate(pairs):
        try:
            a=load_axis(ap,'adt');g=load_axis(gp,'glo')
            if len(a)<min_adt or len(g)<min_glo:raise ValueError(f'too few points ADT={len(a)} GLORYS={len(g)}')
            if require_terminal and max(a[:,0])<terminal:raise ValueError(f'ADT terminates before {terminal}E (end={max(a[:,0]):.2f})')
            if np.max(seg_lengths(a))>250 or np.max(seg_lengths(g))>250:raise ValueError('a path has a >250 km segment')
            metr=polyline_rmse(a,g,step)
            daily.append({'date':pd.to_datetime(dt,format='%Y%m%d').strftime('%Y-%m-%d'),**metr,
                          'adt_points':len(a),'glorys_points':len(g),'adt_terminal_lon':float(a[-1,0]),
                          'glorys_terminal_lon':float(g[-1,0]),'adt_axis_file':str(ap),'glorys_axis_file':str(gp)})
            ds=pd.to_datetime(dt,format='%Y%m%d').strftime('%Y-%m-%d')
            sections.extend(section_records(a,ds,'ADT'))
            sections.extend(section_records(g,ds,'GLORYS'))
            if len(paths)<12 and (j%max(1,int(len(pairs)/10))==0):paths.append((ds,a,g))
        except Exception as e:failure.append({'date':dt,'adt_path':str(ap),'glo_path':str(gp),'error':repr(e)})
        if out is not None and (j+1)%100==0:
            pd.DataFrame(daily).to_csv(Path(out)/'daily'/'每日整体路径距离.csv',index=False,encoding='utf-8-sig')
            pd.DataFrame(sections).to_csv(Path(out)/'daily'/'每日六断面交点.csv',index=False,encoding='utf-8-sig')
            print(f'[R70] processed {j+1}/{len(pairs)}, pass={len(daily)}, fail={len(failure)}',flush=True)
    return pd.DataFrame(daily),pd.DataFrame(sections),pd.DataFrame(failure),paths

def make_section_compare(sections,min_days=15,boot_n=400,boot_block=12,seed=1):
    if sections.empty:return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()
    v=sections.copy(); v['date']=pd.to_datetime(v['date']);v['month']=v['date'].dt.to_period('M').astype(str)
    wide=v.pivot(index=['date','section'],columns='source',values='distance_km').reset_index()
    cnt=v.pivot(index=['date','section'],columns='source',values='n_intersections').reset_index()
    ambiguous=cnt.rename(columns={'ADT':'adt_crossings','GLORYS':'glorys_crossings'})
    wide=wide.merge(ambiguous,on=['date','section'],how='left')
    wide['diff_km']=wide['GLORYS']-wide['ADT'];wide['month']=wide['date'].dt.to_period('M').astype(str)
    common=wide.dropna(subset=['ADT','GLORYS']).copy()
    common['adt_anchor_km']=common.groupby('section')['ADT'].transform('mean')
    common['adt_anomaly_km']=common['ADT']-common['adt_anchor_km']
    common['glorys_anomaly_km']=common['GLORYS']-common['adt_anchor_km']
    group=common.groupby(['section','month'],as_index=False).agg(adt_km=('ADT','mean'),glorys_km=('GLORYS','mean'),
                                    n=('date','nunique'),anchor=('adt_anchor_km','first'),
                                    n_ambig=('adt_crossings',lambda x:int(np.sum(np.asarray(x)>1))))
    group=group[group['n']>=min_days].copy()
    group['adt_anomaly_km']=group['adt_km']-group['anchor']
    group['glorys_anomaly_km']=group['glorys_km']-group['anchor']
    group['diff_km']=group['glorys_km']-group['adt_km']
    tr=[]
    for j,sec in enumerate(SECTIONS):
        x=group[group['section']==sec].sort_values('month')
        if len(x)<24:continue
        t=((pd.PeriodIndex(x['month'],freq='M').to_timestamp()-pd.Timestamp('1993-01-01')).days.to_numpy()/365.2425)
        a=x['adt_anomaly_km'].values;g=x['glorys_anomaly_km'].values
        fit=trend_paired_boot(t,a,g,boot_block,boot_n,seed+j)
        corr=float(np.corrcoef(a,g)[0,1]) if np.std(a)>0 and np.std(g)>0 else np.nan
        tr.append({'section':sec,'months':len(x),'correlation':corr,
                   'monthly_rmse_km':float(np.sqrt(np.mean((a-g)**2))),
                   'monthly_bias_km':float(np.mean(g-a)),**fit})
    return common,group,pd.DataFrame(tr)

def forecast_compare(manifest_path,adt_axis_dir,output=None):
    p=Path(manifest_path)
    if not p.is_file():return pd.DataFrame(),pd.DataFrame(), 'Forecast manifest absent: monitoring validation only; NOT a forecast verification.'
    manifest=pd.read_csv(p,dtype={'model':str})
    req=['model','seed','issue_date','target_date','lead_days','path_file']
    if any(k not in manifest.columns for k in req):raise ValueError(f'Forecast manifest requires {req}')
    rows=[];secs=[];fail=[]
    for r in manifest.itertuples(index=False):
        target=pd.Timestamp(r.target_date).strftime('%Y%m%d');issue=pd.Timestamp(r.issue_date)
        if (pd.Timestamp(r.target_date)-issue).days!=int(r.lead_days):
            fail.append({'target_date':target,'reason':'issue+lead != target'});continue
        adt=Path(adt_axis_dir)/f'{target}.npz'
        pf=Path(r.path_file);pf=pf if pf.is_absolute() else p.parent/pf
        if not adt.exists() or not pf.exists():
            fail.append({'target_date':target,'reason':f'missing ADT={adt.exists()} forecast={pf.exists()}'});continue
        try:
            ref=load_axis(adt,'adt');branches=load_axis(pf,'pred')
            if isinstance(branches,np.ndarray):branches=[branches]
            for b in branches:
                if len(b)<3:raise ValueError('invalid forecast branch')
            # Do NOT connect Branch 1 and Branch 2 with an invented straight segment.
            rep=np.vstack([densify(b) for b in branches])
            # Regionally report directed predicted->ADT, ref->pred without artificially joining branches.
            met=polyline_rmse(rep,ref,5.) if len(branches)==1 else polyline_multi_rmse(branches,ref)
            rr={'model':r.model,'seed':int(r.seed),'issue_date':str(issue.date()),
                'target_date':pd.Timestamp(r.target_date).strftime('%Y-%m-%d'),'lead_days':int(r.lead_days),**met}
            rows.append(rr)
            for sec in SECTIONS:
                seg=SECTIONS[sec]
                target_cross=intersections(ref,seg)
                pred_cross=[]
                for b in branches:pred_cross.extend(intersections(b,seg))
                # Pre-registered: first crossing in branch 1, then branch 2; no ADT-dependent selection.
                a=target_cross[0][2] if target_cross else np.nan
                pp=pred_cross[0][2] if pred_cross else np.nan
                secs.append({**{k:rr[k] for k in ['model','seed','issue_date','target_date','lead_days']},
                             'section':sec,'adt_distance_km':a,'pred_distance_km':pp,
                             'error_km':pp-a,'abs_error_km':abs(pp-a),
                             'adt_crossings':len(target_cross),'pred_crossings':len(pred_cross)})
        except Exception as e:fail.append({'target_date':target,'reason':str(e)})
    if output is not None:pd.DataFrame(fail).to_csv(Path(output)/'reports'/'预测验证缺失或拒绝记录.csv',index=False,encoding='utf-8-sig')
    return pd.DataFrame(rows),pd.DataFrame(secs),f'Forecast pairs {len(rows)}, failed {len(fail)}'

def polyline_multi_rmse(branches,reference,step=5.):
    # Distances to union of independent branch polylines, without invented joins.
    dense=[densify(b,step) for b in branches]
    # Reference is limited to common represented longitudes, preserving geographic coverage audit.
    lo=min(float(np.nanmin(b[:,0])) for b in dense);hi=max(float(np.nanmax(b[:,0])) for b in dense)
    full=densify(reference,step)
    full=full[(full[:,0]>=lo)&(full[:,0]<=hi)]
    if len(full)<3:raise ValueError('reference has insufficient forecast-region coverage')
    aa=sphere_xyz(np.vstack(dense));bb=sphere_xyz(full)
    ab=chord_to_gc_km(cKDTree(bb).query(aa)[0]);ba=chord_to_gc_km(cKDTree(aa).query(bb)[0]);v=np.r_[ab,ba]
    return {'rmse_km':float(np.sqrt(np.mean(v*v))),'mae_km':float(v.mean()),
            'p90_km':float(np.quantile(v,.9)), 'bias_AtoB_km':float(ab.mean()),'bias_BtoA_km':float(ba.mean())}

def setup_fonts():
    import matplotlib.font_manager as fm
    names={x.name for x in fm.fontManager.ttflist}
    for name in ['Noto Sans CJK SC','Microsoft YaHei','WenQuanYi Micro Hei','SimHei','Source Han Sans SC']:
        if name in names:return name,True
    return 'DejaVu Sans',False

def make_figures(daily,monthly,trends,samples,out,forecast=None):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    out=Path(out)
    font,zh=setup_fonts();plt.rcParams['font.family']=font;plt.rcParams['axes.unicode_minus']=False
    if not daily.empty:
        d=daily.copy();d['date']=pd.to_datetime(d['date'])
        fig,ax=plt.subplots(figsize=(12,3.6),dpi=170)
        ax.plot(d['date'],d['rmse_km'],linewidth=.5,color='#476989',alpha=.6,label='Daily symmetric RMSE')
        ax.plot(d['date'],d['rmse_km'].rolling(60,min_periods=10).mean(),lw=1.8,color='#b56f39',label='60-day mean')
        ax.set_ylabel('Axis separation (km)');ax.set_xlabel('Date');ax.grid(alpha=.2);ax.legend();fig.tight_layout()
        fig.savefig(out/'figures_en'/'Fig01_Daily_Axis_Distance.png',dpi=240);fig.savefig(out/'figures_en'/'Fig01_Daily_Axis_Distance.pdf');plt.close(fig)
    if not monthly.empty:
        fig,axs=plt.subplots(6,1,figsize=(13,12.5),sharex=True,dpi=140)
        for ax,sec in zip(axs,SECTIONS):
            p=monthly[monthly.section==sec].sort_values('month')
            if p.empty:continue
            t=pd.to_datetime(p['month'].astype(str))
            ax.plot(t,p['adt_anomaly_km'],label='ADT-derived axis',lw=.85,color='#234e70')
            ax.plot(t,p['glorys_anomaly_km'],label='GLORYS 50-m axis',lw=.8,color='#c27a43')
            ax.axhline(0,color='#333333',lw=.5);ax.set_ylabel(f'{sec}  (km)');ax.grid(alpha=.18)
        axs[0].legend(ncol=2,fontsize=9);axs[-1].set_xlabel('Month')
        fig.suptitle('Section anomalies centered on the same ADT reference mean',y=.998,fontsize=12)
        fig.tight_layout();fig.savefig(out/'figures_en'/'Fig02_Six_Sections_Monthly_Anomalies.png',dpi=240)
        fig.savefig(out/'figures_en'/'Fig02_Six_Sections_Monthly_Anomalies.pdf');plt.close(fig)
    if trends is not None and not trends.empty:
        fig,ax=plt.subplots(figsize=(9,4),dpi=160)
        x=np.arange(len(trends));w=.32
        ax.bar(x-w/2,trends.adt_km_decade,w,label='ADT',color='#234e70')
        ax.bar(x+w/2,trends.glorys_km_decade,w,label='GLORYS',color='#c27a43')
        ax.set_xticks(x,trends.section);ax.set_ylabel('Linear trend (km/decade)');ax.axhline(0,c='black',lw=.7);ax.legend();ax.grid(axis='y',alpha=.2)
        fig.tight_layout();fig.savefig(out/'figures_en'/'Fig03_Trends_Six_Sections.png',dpi=240)
        fig.savefig(out/'figures_en'/'Fig03_Trends_Six_Sections.pdf');plt.close(fig)
    if samples:
        n=min(len(samples),8);fig,axs=plt.subplots(2,4,figsize=(15,7.7),dpi=120)
        for ax,(d,a,g) in zip(axs.flat,samples[:n]):
            ax.plot(a[:,0],a[:,1],lw=1.5,c='#234e70',label='ADT')
            ax.plot(g[:,0],g[:,1],lw=1.2,c='#c27a43',label='GLORYS')
            for v in SECTIONS.values(): ax.plot([v[0][0],v[1][0]],[v[0][1],v[1][1]],c='0.5',lw=.4)
            ax.set_title(d,fontsize=10);ax.set_xlim(120,147);ax.set_ylim(21,40);ax.grid(alpha=.12)
        for ax in list(axs.flat)[n:]:ax.axis('off')
        axs[0,0].legend(fontsize=8);fig.tight_layout()
        fig.savefig(out/'figures_en'/'Fig04_Axis_Overlays.png',dpi=240);fig.savefig(out/'figures_en'/'Fig04_Axis_Overlays.pdf');plt.close(fig)
    if forecast is not None and not forecast.empty:
        f=forecast.groupby(['model','lead_days'],as_index=False)['rmse_km'].mean()
        fig,ax=plt.subplots(figsize=(7,4))
        for model,x in f.groupby('model'):ax.plot(x.lead_days,x.rmse_km,marker='o',label=model)
        ax.set_xlabel('Forecast lead (days)');ax.set_ylabel('ADT reference RMSE (km)');ax.grid(alpha=.2);ax.legend()
        fig.tight_layout();fig.savefig(out/'figures_en'/'Fig05_Forecast_vs_ADT.png',dpi=240);plt.close(fig)
    # 独立排版中文版本，绝不把英文图直接改名为“中文图”。
    if zh:
        if not daily.empty:
            d=daily.copy();d['date']=pd.to_datetime(d['date'])
            fig,ax=plt.subplots(figsize=(12,3.6),dpi=170)
            ax.plot(d['date'],d['rmse_km'],lw=.5,c='#476989',alpha=.65,label='逐日对称路径RMSE')
            ax.plot(d['date'],d['rmse_km'].rolling(60,min_periods=10).mean(),lw=1.8,c='#b56f39',label='60天滑动均值')
            ax.set_ylabel('流轴分离距离（km）');ax.set_xlabel('日期');ax.set_title('独立ADT流轴与GLORYS 50米流轴距离')
            ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(out/'figures_cn'/'图1_逐日ADT与GLORYS流轴距离.png',dpi=240)
            fig.savefig(out/'figures_cn'/'图1_逐日ADT与GLORYS流轴距离.pdf');plt.close(fig)
        if not monthly.empty:
            fig,axs=plt.subplots(6,1,figsize=(13,12.5),sharex=True,dpi=140)
            for ax,sec in zip(axs,SECTIONS):
                p=monthly[monthly.section==sec].sort_values('month')
                if p.empty:continue
                t=pd.to_datetime(p['month'].astype(str))
                ax.plot(t,p['adt_anomaly_km'],label='ADT参考流轴',lw=.85,c='#234e70')
                ax.plot(t,p['glorys_anomaly_km'],label='GLORYS流轴',lw=.8,c='#c27a43')
                ax.axhline(0,c='0.2',lw=.5);ax.set_ylabel(f'{sec}断面（km）');ax.grid(alpha=.18)
            axs[0].legend(ncol=2,fontsize=9)
            axs[-1].set_xlabel('月份')
            fig.suptitle('六断面径向距离距平（统一ADT参考均值）',y=.998)
            fig.tight_layout();fig.savefig(out/'figures_cn'/'图2_六断面月尺度距平对比.png',dpi=240)
            fig.savefig(out/'figures_cn'/'图2_六断面月尺度距平对比.pdf');plt.close(fig)
        if trends is not None and not trends.empty:
            fig,ax=plt.subplots(figsize=(9,4),dpi=160);x=np.arange(len(trends));w=.32
            ax.bar(x-w/2,trends.adt_km_decade,w,label='ADT参考流轴',color='#234e70')
            ax.bar(x+w/2,trends.glorys_km_decade,w,label='GLORYS识别流轴',color='#c27a43')
            ax.set_xticks(x,trends.section);ax.set_ylabel('趋势（km/十年）');ax.set_title('六断面长期趋势对比')
            ax.axhline(0,c='black',lw=.7);ax.legend();ax.grid(axis='y',alpha=.2)
            fig.tight_layout();fig.savefig(out/'figures_cn'/'图3_六断面长期变化趋势.png',dpi=240)
            fig.savefig(out/'figures_cn'/'图3_六断面长期变化趋势.pdf');plt.close(fig)
        if samples:
            n=min(len(samples),8);fig,axs=plt.subplots(2,4,figsize=(15,7.7),dpi=120)
            for ax,(d,a,g) in zip(axs.flat,samples[:n]):
                ax.plot(a[:,0],a[:,1],lw=1.5,c='#234e70',label='ADT')
                ax.plot(g[:,0],g[:,1],lw=1.2,c='#c27a43',label='GLORYS')
                for v in SECTIONS.values():ax.plot([v[0][0],v[1][0]],[v[0][1],v[1][1]],c='0.5',lw=.4)
                ax.set_title(d,fontsize=10);ax.set_xlim(120,147);ax.set_ylim(21,40);ax.grid(alpha=.12)
            for ax in list(axs.flat)[n:]:ax.axis('off')
            axs[0,0].legend(fontsize=8);fig.suptitle('典型日期流轴空间对比')
            fig.tight_layout();fig.savefig(out/'figures_cn'/'图4_典型日流轴空间对照.png',dpi=240)
            fig.savefig(out/'figures_cn'/'图4_典型日流轴空间对照.pdf');plt.close(fig)
        if forecast is not None and not forecast.empty:
            f=forecast.groupby(['model','lead_days'],as_index=False)['rmse_km'].mean()
            fig,ax=plt.subplots(figsize=(7,4))
            for model,x in f.groupby('model'):ax.plot(x.lead_days,x.rmse_km,marker='o',label=model)
            ax.set_xlabel('预测提前期（天）');ax.set_ylabel('相对ADT参考流轴的RMSE（km）')
            ax.set_title('独立ADT参考下的路径预测误差');ax.grid(alpha=.2);ax.legend()
            fig.tight_layout();fig.savefig(out/'figures_cn'/'图5_预测流轴与ADT误差.png',dpi=240)
            fig.savefig(out/'figures_cn'/'图5_预测流轴与ADT误差.pdf');plt.close(fig)
    return font,zh

def archive(out,zip_path,stage='completed'):
    root=Path(out); dest=Path(zip_path);dest.parent.mkdir(parents=True,exist_ok=True)
    manifest=[]
    for f in sorted(root.rglob('*')):
        if not f.is_file() or f==dest or f.name.endswith('.zip'):continue
        bb=f.read_bytes();manifest.append({'file':str(f.relative_to(root)),'bytes':len(bb),'sha256':hashlib.sha256(bb).hexdigest()})
    (root/'reports'/'归档文件SHA256清单.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for f in sorted(root.rglob('*')):
            if f.is_file() and not f.name.endswith('.zip'):z.write(f,f.relative_to(root))
    with zipfile.ZipFile(dest) as z:bad=z.testzip()
    if bad:raise RuntimeError(f'ZIP corrupt: {bad}')
    sha=hashlib.sha256(dest.read_bytes()).hexdigest();dest.with_suffix('.zip.sha256').write_text(f'{sha}  {dest.name}\n',encoding='utf-8')
    return dest,sha

def save_df(df,p):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);df.to_csv(p,index=False,encoding='utf-8-sig')



In [ ]:
# ===== Cell 6：ADT/GLORYS每日配对和完整流轴空间误差 =====
ensure(OUT)
pairs,adt_index,glo_index=paired_series_daily(ADT_AX,GLORYS,cfg.START_DATE,cfg.END_DATE,smoke_n=0)
summary={'all_calendar_days':len(expected),'adt_nc_files':len(manifest),
         'adt_successful_tracks':len(adt_index),'glorys_tracks':len(glo_index),
         'paired_monitoring_days':len(pairs),'protocol':'ADT_V6_PythonPort; independent satellite product surface vs GLORYS 50m'}
print('[PAIRED]',summary)
if not pairs:raise RuntimeError('尚无ADT与GLORYS同日有效轨迹：请检查Cell4和目录；不能生成伪造图表')
daily,section_daily,pair_fail,examples=process_pairs(pairs,20,20,True,cfg.MIN_TERMINAL_LON,5.,OUT)
save_df(daily,OUT/'daily'/'每日ADT_vs_GLORYS完整流轴空间误差.csv')
save_df(section_daily,OUT/'daily'/'每日六断面交点和多交点信息.csv')
save_df(pair_fail,OUT/'reports'/'六断面及整体配对失败日期.csv')
print('Monitoring evaluated:',len(daily),'paired errors:',len(pair_fail))
if not daily.empty:
    display(daily[['rmse_km','mae_km','p90_km']].describe().round(2))


In [ ]:
# ===== Cell 7：六断面距平、月尺度趋势、成对差异不确定性 =====
section_pair,section_monthly,section_trends=make_section_compare(
    section_daily,min_days=cfg.MIN_MONTH_DAYS,boot_n=cfg.BOOTSTRAP_N,boot_block=12,seed=cfg.RANDOM_SEED)
save_df(section_pair,OUT/'daily'/'六断面逐日ADT与GLORYS成对误差.csv')
save_df(section_monthly,OUT/'monthly'/'六断面ADT统一基准月尺度距平.csv')
save_df(section_trends,OUT/'tables'/'六断面长期趋势与95CI.csv')
if not section_trends.empty:display(section_trends.round(3))
else:print('WARNING: valid monthly paired data <24 months; trend cannot yet be estimated')
print('NOTE 1: Here only years 1993–2023 with valid paired sections, not 1993–2024.')
print('NOTE 2: ADT and GLORYS use SAME ADT reference mean; ambiguous intersections are separately counted.')


In [ ]:
# ===== Cell 8：预测轨迹的独立验证（有真实预测文件就自动处理；无文件明确未完成） =====
# 可沿用过去R6.5模型权重生成的 predictions_manifest.csv；该清单应包含model/seed/issue_date/target_date/lead_days/path_file。
def locate_forecast_manifest():
    candidates=[]
    if cfg.FORECAST_MANIFEST:candidates.append(Path(cfg.FORECAST_MANIFEST))
    candidates += [ROOT/'R70_ADT_Independent'/'predictions_manifest.csv',
                   RUN/'predictions_manifest.csv',ROOT/'R70_ADT_Independent_Python'/'predictions_manifest.csv']
    return next((p for p in candidates if p.exists()),candidates[0])
PRED_MANIFEST=locate_forecast_manifest()
forecast_daily,forecast_sections,forecast_status=forecast_compare(PRED_MANIFEST,ADT_AX,OUT)
save_df(forecast_daily,OUT/'daily'/'R65单模型预测vs独立ADT逐样本.csv')
save_df(forecast_sections,OUT/'daily'/'R65单模型预测vs独立ADT六断面.csv')
summary['forecast_status']=forecast_status
summary['forecast_sample_count']=len(forecast_daily)
print('[PREDICTION]',forecast_status,'manifest:',PRED_MANIFEST)
if len(forecast_daily):display(forecast_daily.groupby(['model','lead_days'])[['rmse_km','mae_km']].mean().round(2))
else:
    print('预测验证未完成：未检测到可配对的真实预测坐标文件。请将现有R6.5权重导出的预测npz+manifest放入上述路径后，重运行本单元及其后面的图表/归档单元。')


In [ ]:
# ===== Cell 9：可选原MATLAB与Python对照，不影响无MATLAB的独立运行 =====
if cfg.COMPARE_MAT_REFERENCE:
    src=Path(cfg.MATLAB_REFERENCE_DIR)
    if not src.is_dir():print('[MATLAB CROSSCHECK] dir missing, no fabricated agreement:',src)
    else:
        rows=[]
        for p in list(src.glob('*.mat'))[:32]:
            mt=re.search(r'(\d{8})\.mat$',p.name)
            if not mt:continue
            py=ADT_AX/(mt.group(1)+'.npz')
            if not py.exists():continue
            try:
                a=load_axis(p,'adt');b=load_axis(py,'adt')
                rows.append({'date':mt.group(1),**polyline_rmse(a,b)})
            except Exception as e:rows.append({'date':mt.group(1),'error':repr(e)})
        save_df(pd.DataFrame(rows),OUT/'reports'/'MATLAB_V6_vs_Python_V6交叉比对.csv')
        print('MATLAB/Python paired checks:',len(rows),'not a substitute for broad physical validation')
else:print('No optional MATLAB reference required. Numerical equivalence not asserted.')


In [ ]:
# ===== Cell 10：根据真实实验结果绘图，中文字体不可用时只输出英文图、记录限制 =====
font,has_cjk=make_figures(daily,section_monthly,section_trends,examples,OUT,forecast_daily)
print('Font:',font,'Chinese available:',has_cjk)
if not has_cjk:print('中文字体不可用，请在WSL安装 fonts-noto-cjk；英文论文图已生成，不输出乱码中文图。')


In [ ]:
# ===== Cell 11：最终状态、异常审计、自动ZIP与SHA256、WSL/Windows路径 =====
from collections import Counter
status_counts=adt_tracking_report.status.value_counts(dropna=False).to_dict() if not adt_tracking_report.empty else {}
summary.update({'run_full':cfg.RUN_FULL,'adt_status_counts':status_counts,'n_months_with_common_pairs':len(section_monthly),
                'n_six_section_trends':len(section_trends),'n_pair_failures':len(pair_fail),
                'cjk_figures_generated':bool(has_cjk),'adt_data_origin':'CMEMS DUACS 0.125deg daily ADT',
                'model_adt_independence_note':'ADT is a satellite altimetry reference, not error-free truth; GLORYS may assimilate altimetry',
                'tracker_method_equivalence':'Python numerical port; exact MATLAB equality unverified'})
(REP/'R70_PY_最终实验状态.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
(OUT/'reports'/'R70_PY_实验边界及投稿限制.txt').write_text("""ADT是卫星高度计独立来源参考，不是无误差的绝对真实流轴。GLORYS同化高度计可能降低统计独立性。ADT表层梯度脊线与50m流速核存在物理定义差异。
Python V6是对原MATLAB V6机制的移植，但未完成逐日双实现严格数值一致性对照。
监测≠预测；不存在真实预测轨迹时不能将监测误差冒充预测结果。
六断面首次交点是预注册选择，出现多个交点的日期会统计；ADT未通过完整路径QC的日期不会被静默使用。
1993–2023是本次ADT数据能覆盖的完整时段；2024不可冒用。
""",encoding='utf-8')
ZIP=ROOT/'R70_PY_上传给ChatGPT_ADT独立参考验证完整结果.zip'
result_zip,sha=archive(OUT,ZIP,'complete_or_partial')
# Include lightweight tracking reports and QC protocol alongside evaluation ZIP as companion.
qc_zip=ROOT/'R70_PY_上传给ChatGPT_日期覆盖与流轴追踪QC.zip'
with zipfile.ZipFile(qc_zip,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in REP.glob('*.csv'):z.write(p,p.name)
    for p in REP.glob('*.json'):z.write(p,p.name)
qc_hash=hashlib.sha256(qc_zip.read_bytes()).hexdigest()
qc_zip.with_suffix('.zip.sha256').write_text(qc_hash+'  '+qc_zip.name+'\n',encoding='utf-8')
print('RESULT ZIP:',result_zip,'SHA256',sha)
print('QC ZIP:',qc_zip,'SHA256',qc_hash)
print('Windows: D:\\KuroshioDaily1993_2025\\'+ZIP.name)
display(FileLink(str(result_zip)));display(FileLink(str(qc_zip)))
print('实验状态：监测评估完成' if cfg.RUN_FULL and len(daily)>500 else '注意：仅冒烟/部分监测结果，还不是完整多年验证')
if len(forecast_daily)==0:print('预测验证状态：未完成（仍需现有单模型的真实逐起报路径坐标）')
